<img src=../figures/Brown_logo.svg width=50%>

## Data-Driven Design & Analyses of Structures & Materials (3dasm)

## Lecture 19.2

### Elvis Aguero | <a href = "mailto: elvis_alexander_aguero_vera@brown.edu">elvis_alexander_aguero_vera@brown.edu</a>  | PhD candidate

## Introduction

**What:** A lecture of the "3dasm" course. What happened when we pointed **adda** — the agentic
framework from Lecture 19.1 — at a real, open research problem.

**Where:** This notebook comes from this [repository](https://github.com/bessagroup/3dasm_course)

**Reference for this lecture:** Bessa, M. A., Glowacki, P., & Houlder, M. (2019). *Bayesian
Machine Learning in Metamaterial Design: Fragile Becomes Supercompressible.* Advanced Materials,
31(48), 1–6. [doi:10.1002/adma.201904845](https://doi.org/10.1002/adma.201904845)

**How:** This is a **research diary**, not a product demo. The numbers are real, most of the
ideas failed, and the honest headline is at the end.

Speaker notes.

Set expectations in the first minute. If students expect a triumphant "the AI discovered a new
metamaterial", they will misread everything that follows. The interesting result is not a
winning design — it is a *map of a design space*, produced quickly, with honest verdicts,
including on ideas we were personally attached to.

The single most important sentence in this lecture is on the "did it beat the human" slide near
the end: the best design found is the one a human found first. Say it plainly and without
embarrassment.

## Outline for today

* The problem, and the baseline we had to beat
* What we asked for — including the clause that made it hard
* What happened: 23 runs, 28 design ideas, six weeks
* Five ideas, in detail — including one of *our own* that failed
* **The headline it almost reported** (and how it caught itself)
* The traps that produced wrong verdicts
* The twelve times *we* changed the rules underneath it
* Where the human was indispensable
* The honest scoreboard

**Reading material**: this notebook + Bessa, Glowacki & Houlder (2019).

# The problem

## Supercompressible metamaterials

A slender lattice — a "rocking mast" — of three longerons joined by a top and bottom ring.

Compress it and it **coils** rather than breaking: the structure buckles into a stable, tightly
wound configuration, and springs back. *Fragile becomes supercompressible.*

<br>

The design problem: choose the geometry that maximizes the **normalized critical buckling
load**, among designs that are actually coilable.

## The baseline

<img src=./img/bessa_baseline_native.gif width=32% align='right'>

The reference point from Bessa, Glowacki & Houlder (2019):

* **Design:** circular longeron cross-section, 3 longerons, 1 storey, circular top and bottom rings
* **Geometry:** `ratio_d = 0.02005`, `ratio_pitch = 0.25`, `ratio_top_diameter = 0.2505`
* **Result:** σ<sub>cr,nd</sub> = **0.1306 kPa/longeron**, max local strain = 0.0198,
  fully reversible coiling

<br>

Every "× Bessa" figure in this lecture is a multiple of that number.

Speaker notes.

The animation is a native Abaqus/CAE viewer export of this exact design, re-solved specifically
for the provenance deck. It is not a schematic and not a reconstruction — colour is axial strain
from the simulation's own field output.

One caveat to flag if a sharp student asks: this baseline was later re-measured under a
corrected oracle as 0.1122 kPa. We get to that on the contract-change slide.

## What counts as a valid design

Two feasibility criteria, from the original paper's own definitions:

<br>

| criterion | requirement |
| :-- | :-- |
| **maximum compression strain** (`mcs`) | at least **80%** — it must actually coil most of the way down |
| **maximum local strain** (`mls`) | at most **2%** — on *any* strain component, shear included |

<br>

A design with a spectacular buckling load that violates either one is **not an answer.**

This sounds like bookkeeping. It is the single most consequential part of the brief — and as we
will see, almost every apparent breakthrough in six weeks died on one of these two lines.

## What we asked for

> Find a rocking-mast lattice design with **maximum normalized critical buckling load** among
> coilable topologies, **surpassing Bessa et al. (2019)** using topology parameters the original
> paper left unexplored.

<br>

Design space: **13-dimensional** — 4 topology integers + 9 cross-section parameters.

And then the clause that made it a research problem rather than an optimization exercise:

> **Novelty must be a new *shape or arrangement*, not a resized cross-section.**

Why that clause matters: Bayesian optimization already solves a 13-dimensional box. If all we
wanted was the best point inside a fixed parametrization, we would not need an agent — we would
need a weekend of compute.

**The frontier is inventing the parametrization.** So we asked for that instead.

Speaker notes.

This is the intellectual pivot of the whole project and it came from the PI. Prescribing a fixed
box and asking the agent to beat a baseline by some percentage wastes the one thing an LLM
uniquely brings: it has read the literature and can propose a *representation* nobody wrote down
for this problem. So the task was reframed — here is the baseline; propose new designs and study
each for signs of a breakthrough.

The cost of that reframe is that "did it win" becomes much harder to answer, because a number
can now be real and still not count. That difficulty is the subject of most of this lecture.

# What happened

## Six weeks, in numbers

<br>

| | |
| :-- | :-- |
| closed runs | **23** |
| genuinely new design ideas tested | **28** (D1 – D28) |
| period | 2026-06-29 → 2026-08-09 |
| cost per run | **\$20.68 – \$54.45** at comparable wall clock |
| oracle | Abaqus linear buckling (Stage 1) + Riks post-buckling (Stage 2), on SLURM |
| designs evaluated in the single largest campaign | **330** |

<br>

Every run ended in a critic-reviewed, reproduction-gated notebook, or it did not end.

The record of all of it is a single append-only deck: **one slide per genuinely new idea,
regardless of verdict.**

A family that *failed* is exactly what stops a future run from re-proposing it.

## How each idea was recorded

Five fields, fixed format, every one of them:

<br>

| field | what it must contain |
| :-- | :-- |
| **What** | precisely what was tried, plus the free variables and their sampled bounds |
| **Origin** | where the idea came from — a real citation, **or the honest** *"common sense"* |
| **Stats** | the funnel: `n → coilable → Riks-converged → feasible`, plus quartiles |
| **Verdict** | a Popperian **STATUS** *and* a separate **PRACTICAL** answer |
| **Seed** | can this *idea* still generate a new design? `FERTILE` / `BARREN` |

Two of these deserve attention.

**Origin forbids inventing a citation.** If only a named theory was cited without a delegation
verifying a specific paper, the field must say so — *"named theory, no single paper verified this
run"* — rather than fabricate an author and year to look more rigorous than the evidence is.

**Verdict splits two independent axes**, because one word was hiding the answer:

* **STATUS** — `SUPPORTED` / `FALSIFIED` / `INCONCLUSIVE` (the charter's vocabulary)
* **PRACTICAL** — `WORKS` / `DEAD-END` / `WEAK` / `MIXED` / `UNTESTABLE`

A formal `INCONCLUSIVE` often sat on top of a perfectly clear practical dead end. A reader had to
parse a paragraph to learn whether the idea actually worked.

## Why `Seed` exists

`Verdict` answers *"did this experiment work?"*. It cannot answer:

> **Can this *idea* still generate a new design?**

<br>

Those are different questions, and conflating them causes two opposite failures: an agent
re-running a settled search, and an agent spending a whole campaign optimizing something that
could never count.

The distinction, worked:

* **D25 tape spring** — the 6-D arc box is exhausted (406 evaluations before contact, 68 under
  it). But a *twisted* open shell is a different design. **FERTILE.**
* **D6 rectangle** — barren **not because it was searched hard**, but because every variant is
  still a cross-section change at fixed topology. No number it produces can clear the novelty
  bar. **BARREN.**

<br>

So `BARREN` is a claim about an idea's **ceiling**, never about how much evidence was gathered.

Speaker notes.

This is a genuinely new distinction that fell out of doing the work, and it is the one piece of
methodology here I would defend as transferable to any search-heavy research programme, agentic
or not. "We searched it hard and found nothing" and "nothing in this family could ever count"
are completely different states of knowledge, and research groups routinely record only the
first while acting as though they had established the second.

# Five ideas

## D1 · Pretwisted longerons — the first idea

<br>

* **What:** a helical pre-twist on each longeron, on top of the full 7-D cross-section search.
  Could twisting the legs reach a higher coiling-mode eigenvalue?
* **Origin:** *common sense mechanistic hypothesis — not a literature citation.* (Stated as such.)
* **Stats:** n = 46 → **6 coilable** → 0 Riks → **0 feasible.** Every coilable design at or
  below the un-twisted baseline.
* **Verdict:** `INCONCLUSIVE · DEAD-END`

Why `INCONCLUSIVE` and not `FALSIFIED`? Because a **license-server outage killed 26 of the
planned runs**, so the test fell short of its own registered ≥80-evaluation bar.

The charter does not let you upgrade a verdict because the trend looked obvious. The 46
completed evaluations point one way with no ambiguity — and the *formal* status still records
that the test was not adequate.

**No animation on this slide.** No design in the pre-twist family ever cleared the coilability
bar, so there is no winning geometry to render — and the deck's rule is to show a *typical*
design or nothing, never an empty slot dressed up as a result.

## D6 · Anisotropic rectangle, reversed orientation

<img src=./img/run17_rectangle_native.gif width=30% align='right'>

* **What:** rectangular longeron, radial **short** / tangential **long** — the reverse of an
  orientation this same run had already falsified.
* **Origin:** direct extension of the elliptical-substitution idea; common sense, not literature.
* **Stats:** n = 165 → 149 coilable → 148 Riks → **6 feasible (2.79× Bessa)**
  <br>best: `a=.0092 b=.0188 pitch=.602 top_d=.038` → σ = **0.3644**, mcs = 1.00, mls = 0.0195
* **Verdict:** `SUPPORTED · WORKS` — real, repeatable, not a fluke

This became **`run17_rectangle`**, the anchor baseline for the rest of the study — later refined
to **0.7704 kPa, 5.9× Bessa.**

<br>

* **Seed:** `BARREN` — *"it SUCCEEDED and became the floor."*

Beat `run17_rectangle`, not Bessa. The bar moved up, and it moved up because of a design the
system found.

Speaker notes.

D6 is the study's one unambiguous success, and it is worth being clear about what kind of success
it is: a 5.9x improvement on the published baseline, found by systematic search inside a
cross-section family. That is a real and useful engineering result.

It is also, by our own novelty clause, not the thing we asked for — it is a resized and
reoriented cross-section at fixed topology. Hence BARREN. Both statements are true at once, and
holding them together is the skill this lecture is trying to teach.

## D10 · Elliptical rings with a phase offset — *our* idea

<img src=./img/elliptical_rings_native.gif width=30% align='right'>

* **What:** replace the circular top and bottom rings with independently parametrized
  **ellipses**, plus a **phase offset** between their major axes — breaking the rotational
  symmetry that forces every longeron into identical peak curvature.
* **Stats:** n = 67 → **9 coilable** → 9 Riks → **0 feasible**
* **Verdict:** `INCONCLUSIVE · DEAD-END`

This was not the agent's idea. **It was ours** — the worked example in our own design-space
framework document, the concrete illustration of what "invent a new low-dimensional
parametrization" was supposed to mean.

<br>

The finding: `mcs` **collapses from 0.9999 to 0.398 at the first non-circular step tested.**
Elliptical rings very sharply destroy coilability rather than redistributing strain.

The system tested our favourite idea and told us it did not work — with a funnel, a quartile
table, and a mechanism.

<br>

**That is the product.** Not a winner: a *fast, honest no* on an idea we would otherwise have
spent a month on ourselves.

Speaker notes.

Do not rush this slide. It is the most persuasive argument in the lecture for the whole approach,
and it lands precisely because the failed idea was the PI's own and is documented as such in the
framework spec.

Note also the epistemic care in the verdict: INCONCLUSIVE rather than FALSIFIED, because the
guiding constraint surrogates were not demonstrably above chance, so a closed non-existence
verdict is not licensed. The raw picture is about as close to a clean dead end as the study ever
sees short of a formal FALSIFIED — and the status still refuses to overclaim.

## D21 · Tensegrity longerons — **1691× Bessa**

<img src=./img/tensegrity_native.gif width=28% align='right'>

* **What:** replace the bending longeron with a pin-jointed, prestressed **Class-1 tensegrity**
  assembly — stiffness from prestress and geometry, not from beam bending.
* **Origin:** Amendola et al. (2018) on tensegrity prestress stiffness, contrasted against
  Meng (2012) / Sorrentino (2021) on bending-family strain–stiffness coupling. *A real citation.*
* **Stats:** n = 45 → 45 coilable → 44 Riks → **12 feasible (1691× Bessa)**
  <br>best: σ = **220.89 kPa**, mls ≈ **9 × 10⁻¹⁴**

The largest σ<sub>cr,nd</sub> in the entire study. Re-verified by direct extraction from the
simulation output. **Not an error.**

<br>

* **Verdict:** `SUPPORTED (DISQUALIFIED) · DEAD-END`

Look at the strain: **mls ≈ 9 × 10⁻¹⁴.** The material is not straining *at all*.

It reaches 80% compression by rotating rigid bars about pin joints — a **folding linkage**, not
elastic coiling. It is a real number measuring the wrong mechanism.

So we changed the rules: **reaching the compression target by rotating rigid bars, with almost
no material stretch, stopped counting.**

* **Seed:** `BARREN` — and independently blocked on printability (prestressed cables, pin joints)

<br>

A `DISQUALIFIED` verdict is **always** practically `DEAD-END`, regardless of how large the raw
figure looked. Collapsing "big number, doesn't count" into anything else is exactly the confusion
the two-axis verdict exists to prevent.

Speaker notes.

1691x is the number a press release would have led with. It is also completely worthless as an
answer to the question we asked, and the system's own record says so in the same breath as
reporting it.

This is also the clearest example of the human owning the contract. Nothing in the physics told
us folding linkages don't count -- that is a judgement about what we are trying to build. The
agent found a legitimate exploit in an underspecified brief; we closed it.

## D17 · Kresling origami longerons — feasible, then demoted

<img src=./img/kresling_native.gif width=28% align='right'>

* **What:** each longeron becomes two straight segments meeting at an interior hinge, offset
  circumferentially — coupling axial compression to rigid-body strut re-orientation.
* **Origin:** the **Kresling** origami folding pattern — *a real, specific geometric precedent,
  not a fabricated citation.*
* **Stats:** n = 45 → 37 coilable → 37 Riks → **8 feasible (5.44× Bessa)**
  <br>best: σ = **0.7111**, mcs = 1.00, mls = 0.0196
* **Verdict:** `INCONCLUSIVE · DEAD-END`

Read that verdict against those stats. **Eight feasible designs. It cleared all four criteria
that existed at the time.**

<br>

Then we added a fifth criterion — **material passing through a ring's own footprint is a
failure** — and Kresling was demoted.

It was a validated winner for three days.

**This is why verdicts from different weeks are not comparable**, and why the record is
append-only rather than corrected in place.

## D25 · Tape-spring longerons — the biggest campaign

<img src=./img/tape_spring_native.gif width=27% align='right'>

A thin-walled **open circular arc** — the tape measure that snaps flat and coils.

* **What was tested:** two campaigns, **330 designs** under contact — a 64-design pilot and a
  256-design Sobol sweep — plus 10 paired contact-on/contact-off
* **Result:** **0 feasible**, and the binding criterion is unanimous

Of the 28 designs that reached a verdict:

* **28 fail on compression. 0 fail on strain.**
* median reaches **2.2%** compression; the best reaches **21%** — against the **80%** required
* short by a factor of **3.7×**

<br>

They run out of strain budget before they get anywhere.

## D25 — and the parameter that decided it

<img src=./img/restudy_tape_spring_contact.gif width=27% align='right'>

Of six free parameters, exactly **one** moves the blocker: `alpha_tape`, the **arc angle**
(ρ = −0.600, Holm-adjusted p = 0.003; every other parameter adjusts to p = 1.00 — including
thickness).

<br>

And the best designs sit **on the `alpha_tape` lower bound** — the top quarter average 14% of
the range.

Normally "the optimum sits on a bound" means *widen the bound, you haven't measured this
dimension.*

**Here it means the opposite**, and it is the strongest single argument for closing the family:
the search is pushing toward the *shallowest arc allowed*, and widening the bound further does
not find a better tape spring — **it deletes the arc.**

The best design in the whole campaign has 0.5 mm of section depth. It is a **flat strip** —
which is D6 territory, already searched.

> **The optimizer's preferred direction exits the family.**

Speaker notes.

This is my favourite piece of reasoning in the entire study, and it is not a reasoning step
anyone taught the system explicitly. "The optimizer wants to leave the design space" is a
qualitatively different kind of evidence than "we sampled a lot and found nothing", and it is
much stronger. Worth asking the room whether they would have drawn the same conclusion from a
parameter pinned at its bound.

## D28 · Multi-leaf (leaf-spring) longeron — and an accidental control

<img src=./img/leaf_spring_native.gif width=28% align='right'>

* **What:** split each longeron into `n_leaves` thin leaves stacked in the winding plane, free to
  slide — a leaf spring. Total depth carries the load while each leaf bends at its own small
  depth, paying the strain penalty *per leaf*.
* **Stats:** n = 3 → 3 coilable → 2 Riks → 1 feasible
* **Verdict:** `FALSIFIED · DEAD-END`

`n_leaves = 1` is the family's own **regression control** — and it reproduced the incumbent
rectangle to **four significant figures** (σ_peak 0.6071, σ_eig 0.7704, mls 0.0199).

The one genuinely multi-leaf point **regressed 7.7×** (σ_eig 0.770 → 0.100) at 482 s versus 84 s.

The mechanism: stacking leaves splits the depth that carries load **without changing the
curvature that caps it** — so it pays the strain penalty twice.

<br>

* **Seed:** `BARREN` as stated. Decoupling leaf spacing from the winding radius would be a
  *different idea*, needing a different argument than "more leaves".

**A trap this created.** `n_leaves = 1` topped that run's feasible ledger and was briefly
reported as *a new family's best design.*

It is not. **It is the rectangle**, arriving through a different code path.

# The headline it almost reported

## Six designs that appeared to peak late

In one run, six designs looked like they reached their peak load near **50% compression** —
after coiling, which would have been a genuinely new mechanism.

<br>

One of them reported:

<br>

## σ<sub>peak</sub> = 245.8 kPa
### ≈ 400× the reference design

This would have been the headline. It would have been on a slide. It would very likely have been
in a paper.

## The signature

All six shared one thing:

<br>

```
window_n == history_n
```

<br>

The reported "peak" was at **the last computed increment** — and `mcs_at_peak == mcs`.

The solve had **died**, and the final frame before it died was the largest load seen.

It was not a load rise. It was a **truncated response**: the graph stops at its highest point
because it stops, not because it turns over.

In [ ]:
import pandas as pd

# The diagnostic signature, as a table. window_n is the number of increments inside the
# measurement window; history_n is the number the solve actually computed.
designs = pd.DataFrame({
    "design":     ["A", "B", "C", "D", "E", "F", "legit"],
    "sigma_peak": [245.8, 88.4, 41.2, 33.7, 22.9, 18.1, 0.6071],
    "window_n":   [61, 44, 52, 38, 70, 29, 70],
    "history_n":  [61, 44, 52, 38, 70, 29, 73],
})
designs["truncated"] = designs["window_n"] == designs["history_n"]
designs

The last row is the legitimate design: `window_n = 70 < history_n = 73`. **The window closed
before the solve ran out** — so the response really did turn over inside the measured range.

<br>

Three increments of difference separate a real result from a 400× artifact.

## Who caught it

<br>

**The strategizer found that signature itself.**

<br>

It did not bank the number. It registered the suspicion as a hypothesis and spent a delegation
on a **targeted falsification** of its own would-be headline — then reported the six as
artifacts.

Read that again as a claim about the framework rather than about the model: the system was
**built so that "this number is too good" is a testable hypothesis with a place to live**, and
so that closing it required evidence.

Absent the hypothesis ledger, "huh, that's surprising" has nowhere to go except into the
abstract.

Speaker notes.

This is the emotional centre of the lecture. Deliver it slowly.

Be precise about the credit, though. The model noticed an anomaly -- good models do that. What
the *framework* contributed is that noticing had consequences: a hypothesis could be registered,
a delegation could be spent falsifying it, and the charter forbade closing it on a hunch either
way. A capable model with nowhere to put a doubt will usually resolve the doubt in favour of the
exciting answer, because that is what the surrounding text rewards.

Also worth naming honestly: the truncation convention that made this detectable
(window_closed_before_failure) was *our* engineering, added because an earlier run had been
fooled. The system caught this one because we had already been burned.

# The traps
## Four ways the measurement lied

## Trap 1 — A statistic that cannot fail

A campaign's own first printout said:

> *"designs under the 2% strain limit: **36 of 36**"*

<br>

Which sounds like every design passed. It is in fact a **tautology.**

The measurement window **closes at the 2% crossing**. So windowed `mls` is bounded above by 0.02
**by construction** — its maximum cannot exceed the limit it is being compared against.

Comparing it measures **the ceiling, not the design.** It carries no ranking information at all.

The same script then ranked designs by `min(mls)` to find the "closest miss" — which picks the
design that **stalled soonest**. It printed:

```
closest miss: mls=0.000000 mcs=0.0000
```

<br>

The unsaturated statistic is `strain_crossing_mcs`: *at what compression does the design cross
2%?* Phase 1 was re-analysed on it, and the verdict changed.

This trap produced a **wrong verdict twice, in two disguises** — which is why it is written up
as a documented trap rather than only fixed in code.

## Trap 2 — Sentinel zeros that pass a validity check

Of 36 designs recorded as "decided", **eight were empty**: a salvaged solve holding *zero*
usable increments.

<br>

So `window_n = 0` — which satisfies the test `window_n < history_n` **trivially**, and was
recorded as *"the window closed"* on a response containing nothing.

Each reported `mcs = mls = sigma_peak = 0.0`.

They were caught by an oracle **contract guard** — added the same day — on its **first run**, as
a sentinel-zero pattern.

<br>

Corrected denominator: **28**, not 36. The verdict was unchanged. The number reported to a
reader was not.

## Trap 3 — "We migrated it" is not "we tested it"

<img src=./img/restudy_floor_contact.gif width=26% align='right'>

When ground contact was restored, five previously-blocked design families became testable again.

<br>

| design | what was tested | result |
| :-- | :-- | :-- |
| **D25** tape spring | **330 designs** | **Settled** — best reaches 21% vs 80% required |
| **D21** tensegrity | 1 design, contact on/off | floor now stops it; energy +86%, peak stress **unchanged** |
| **D17** Kresling | 1 design | stalls at 75–77%. **Family untested** |
| **D20** laced | 1 design | does not converge. **Family untested** |
| **D26** chiral shell | 1 design | exceeded the time budget. **Family untested** |

Three of those five rows record **only that a code path now exists.**

<br>

> **One design cannot settle a family** — and the one design available is usually the winner of a
> search run *without* contact, which is the **worst possible point to generalize from.**

That slide exists because otherwise a future run reads *"contact is now available"* and
re-proposes all five as fresh ideas.

## Trap 4 — The critic checked the wrong pair

The critic's job included a provenance check: is this headline a genuinely different design, or a
duplicate row?

It cleared the headline as *"a genuinely different, independently-converged design (different
`n_longerons`)"*.

It had compared the headline against a **different run's** `n_longerons = 4` design — **not
against the anchor**, which is also `n_longerons = 3`.

**Wrong pair.**

The notebook's prose reached the right conclusion anyway, so nothing false was published.

But the lesson is not "the critic is unreliable" — it is that **an adversarial reviewer is one
layer, not a guarantee**, and a check that returns "cleared" tells you a comparison was made,
not that it was the right comparison.

Speaker notes.

If a student asks the obvious question -- "so who checks the critic?" -- the honest answer is: we
do, and that does not scale, and it is one of the genuinely unsolved problems here. The
reproduction gate is mechanical and does scale. Judgement about whether the right comparison was
made currently does not.

# The rules moved twelve times
## (and we were the ones moving them)

## The contract changed, repeatedly

Every verdict in this study was decided under the rules in force **that week** — and the rules
moved **at least twelve times.**

<br>

| date | what changed |
| :-- | :-- |
| 07-16 | reference fixed at 0.1306 kPa. Ground contact **removed** — believed an artificial obstruction |
| 07-17 | compression target corrected **90% → 80%**; strain judged only up to that point |
| 07-18 | **folding linkages ruled out** (demoted tensegrity, 1691×) |
| 07-20 | material passing through a ring's footprint became a failure (demoted Kresling, 5.4×) |
| 07-22 | bar lowered to **2×** the reference; the 5.9× result reframed as context |
| 07-23 | strain limit became Bessa's literal wording: 2% on *any* component, shear included |
| 07-28 | a joint-strain scare retracted one headline; a convergence study reconfirmed the 5.9× baseline |
| 08-04 | **novelty must be a new shape or arrangement**, not a resized cross-section |
| **08-06** | ground contact **restored** — *Bessa always had it; we lost it on 07-16* |

**Verdicts from different runs are not directly comparable.** Everything before 2026-08-06 is
the *unversioned era*; today's contract is **v1**, the first one pinned.

## What that costs you

Under **v1**, the reference design measures **0.1122 kPa** — not 0.1306.

<br>

So:

* 2× the reference = **0.2244**
* the 5.9× baseline becomes **0.6077 kPa = 5.42×**

<br>

Every "× Bessa" figure earlier in this lecture is in the **old** metric, consistently so.

And a visible artifact worth naming: **every animation older than 2026-08-06 was rendered
without a floor**, so longerons visibly sink through the base plate.

That is an artifact of the model *as it stood*, not the design's behavior.

Why numbering starts at v1 rather than retroactively labelling the past:

> the pre-08-06 record is **not one contract, it is at least twelve** — and inventing a single
> retroactive version number for it would imply a coherence that did not exist.

## The rule that made this survivable

<br>

> **A contract change never rewrites an existing verdict.**

<br>

Not its numbers, not its animation, not its `STATUS`. A re-test earns a **new** slide; the old
slide keeps everything it had and gains a `(SUPERSEDED)` tag.

`(SUPERSEDED)` reads as **"re-test me"**, not **"I was wrong"** — the evidence was right *and
counted* at the time, and the rules have since changed in a way that could alter the conclusion.

<br>

Because a verdict **was a correct call under the rules of its own time**, and rewriting it
destroys the audit trail a future reader depends on.

Speaker notes.

This is where the "not fully automated" claim becomes concrete and undeniable. Twelve contract
changes in six weeks is not the agent failing to converge -- it is *us* discovering what we
actually meant by the question. Every single one of those rows is a human judgement that no
oracle could have supplied.

The 07-16/08-06 pair is the most humbling: we removed ground contact believing it was an
artificial modelling obstruction, ran for three weeks without it, and then discovered Bessa
always had it and we had introduced a fidelity error. Three weeks of verdicts are affected. Say
this out loud -- it is the strongest evidence in the lecture that the human is still doing
essential, fallible work.

# Where the human was indispensable

## Six things only we could do

**1. Define what counts.** Folding linkages, ring passthrough, the novelty clause. Nothing in
the physics says a prestressed pin-jointed linkage is not an answer — that is a judgement about
what we are trying to build.

**2. Fix our own fidelity errors.** We removed ground contact believing it artificial. It was
not. Three weeks of verdicts were decided in a model that was less faithful than the paper's.

**3. Stop a false boundary from propagating.** A solve-time cap was described in the brief as
*"a hard property"*. It was a **cost** argument. A run closed with 5.7 of 12 hours unspent,
reasoning correctly from a premise we had written wrongly.

## Six things only we could do (cont.)

**4. Write down what must not be inherited.** After one run established a mechanism ceiling
inside the *beam* families it had searched, we had to record explicitly:

> *Do not inherit this as a boundary. "The only escape needs shells with self-contact" locates
> where THIS SEARCH ran out — not where contact-mediated designs live in general. The space of
> unexplored configurations is not indexed by element type.*

**5. Demand the negative record.** One run tested a genuinely new family, falsified it, and its
summary pointed at *"archived"* because nobody had written the slide.

A genuinely new family became **invisible to the next reader.** We made "one slide per new idea
regardless of verdict" a rule the linter now enforces.

**6. Ask for the re-study slide.** So a future run would not read "contact is now available" and
re-propose five settled-looking families.

## So: is it automated?

<br>

## No.

<br>

The contract is ours. The fidelity of the model is ours. What counts as an answer is ours. What
must not be inherited from a previous run is ours.

## But look at what it did do.

## The case for it as a partner

**Breadth at a speed we cannot match.** 28 distinct design families in six weeks, each with a
stated origin, real simulations, a funnel, quartiles, and a verdict. Each one of those would have
been a graduate-student month.

**It killed its own headline.** A 400× result, caught by its own diagnostic, falsified by its own
delegation, reported as an artifact.

**It produced a mechanism, not just a ranking.** One run fitted

<br>

σ<sub>eig</sub> ∝ **J<sup>0.96</sup>**

<br>

— coil-mode critical load is set by the *minimum* of winding-plane bending and torsional
stiffness. Its own summary calls this *"the run's one durable contribution — a mechanism law
with an exponent, not a ranking."*

**It recorded its failures so they stay dead.** `Seed: BARREN` on a family means a future run
does not spend a campaign rediscovering that leaves don't help.

## And it told the truth about itself

<br>

From the deliverable of the final run, unprompted:

<br>

> **the novelty half of the objective was not met.**

<br>

The run that cost \$56.03, ran 306 evaluations over 6.2 hours, passed the critic on its third
attempt, and produced a genuine mechanism law — states in its own write-up that it did not
achieve what it was asked to achieve.

That sentence is worth more than a fabricated win. It is the reason the other 27 verdicts in the
record are worth reading.

Speaker notes.

Land this. A system that reports its own failure to meet the objective is a system whose
successes you can use. The alternative -- a fluent write-up that finds a way to call every run a
partial success -- is worse than useless, because it poisons the record for everyone who reads it
later.

# The honest scoreboard

## Did it beat the human?

<br>

The best feasible design after 23 runs and 28 design families:

<br>

## 0.6071 kPa
### the incumbent rectangle — **rediscovered**

<br>

found as the leaf-spring family's own **regression control.**

Under the v1 contract that is **5.42× the Bessa reference** — a real, large improvement on the
published baseline.

<br>

And it is a design **the search had already found weeks earlier**, arrived at a second time
through a different code path.

**The novelty half of the objective was not met.**

<br>

Twenty-seven other families were proposed, motivated, simulated, and killed.

## Is that a failure?

<br>

Consider what the record now contains that did not exist six weeks earlier:

* **28 design families** with honest verdicts and stated mechanisms
* a **mechanism law** with a fitted exponent
* **330 designs** settling the tape-spring family
* a **map of which directions are barren and which are still fertile**, with named perturbations
* every number traceable to a run, a delegation, and an archived simulation

The value was never going to be a winning design in six weeks. **The value is the map.**

<br>

A negative result that is *trustworthy* and *recorded* is worth more than a positive result that
is neither.

## Summary

* We asked for a **new shape or arrangement** that beats a published metamaterial — not a
  better point in a fixed box.
* Across **23 runs** it proposed and tested **28 design families**, most from real literature,
  and killed almost all of them with mechanisms attached.
* It **caught its own 400× headline** as a truncation artifact and falsified it.
* The measurement lied in four distinct ways; each trap is now documented rather than only fixed.
* **We changed the rules twelve times** — the contract, the fidelity of the model, and what
  counts as an answer were ours throughout, and we got some of them wrong.
* The best design is still the one found early. **The novelty objective was not met, and the
  system says so itself.**

## What to take from these two lectures

<br>

**Not** *"the agent does the research."*

<br>

But: a large fraction of the *labor* of a data-driven design study — and a real share of the
*proposing* — can be delegated, **if** you build the scaffolding that makes self-deception
expensive.

<br>

The scaffolding is the contribution. The charter, the ledger, the gate, and the append-only
record are what make a verdict worth reading — whoever, or whatever, produced it.

And the part that generalizes past agents entirely:

> a claim needs a **registered prediction**, a **severe test**, and a **traceable number** —
> whether the thing making the claim is a language model or a graduate student.